## Properly plot 3d spin configuration

In [ ]:
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib widget

In [ ]:
# load data

directory = "d1.00/run3/"
output_file = directory + "bestconf.dat"

with open(directory + "/result.dat") as f:
    lines = f.readlines()
    grid_length = int(lines[4].strip().split(" ")[-1])
    
data = np.loadtxt(output_file, dtype="int")
unraveled = np.unravel_index(data, (grid_length, grid_length, grid_length))
grid = np.zeros((grid_length, grid_length, grid_length), dtype=int)
grid[unraveled] = 1

In [ ]:
# layer plot

min_layer = min(unraveled[0])
max_layer = max(unraveled[0])

fig = go.Figure()
for i in range(min_layer, max_layer + 1):
    fig.add_trace(go.Heatmap(z=grid[i]))
        
layers = []
for i in range(len(fig.data)):
    step = dict(
        method="update",
        args=[
            {"visible": [False] * len(fig.data)},
            {"title": "Layer: " + str(i)},
        ],  # layout attribute
    )
    step["args"][0]["visible"][i] = True  # Toggle i'th trace to "visible"
    layers.append(step)

sliders = [
    dict(active=0, currentvalue={"prefix": "Layer: "}, pad={"t": 50}, steps=layers)
]

fig.update_layout(sliders=sliders)

fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1,
)

fig.show()

In [ ]:
# 3d plot with Plotly

X, Y, Z = np.mgrid[0: grid_length, 0: grid_length, 0: grid_length]
fig = go.Figure(data=go.Volume(
    x=X.flatten(),
    y=Y.flatten(),
    z=Z.flatten(),
    value=grid.flatten(),
    isomin=0.1,
    isomax=1.0,
    colorscale='ice',
    opacity=0.1, # needs to be small to see through all surfaces
    surface_count=20, # needs to be a large number for good volume rendering
    ))
fig.show()

In [ ]:
# 3d plot with voxels

ax = plt.figure().add_subplot(projection='3d')
ax.voxels(grid, facecolors="blue", alpha=.5) 
# You can add edgecolor="k" in the arguments to outline the individual voxels

ax.axis('auto')
plt.show()